In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle as pkl

from astropy.io import ascii

from astropy.coordinates import ICRS, AltAz, Angle, EarthLocation, SkyCoord, get_sun
import astropy.units as u
from astropy.time import Time
import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.daf.butler import DatasetNotFoundError
import lsst_efd_client

from lsst.ts.ofc import OFC, OFCData

import logging
import smplotlib

In [ ]:
logging.basicConfig(level=logging.DEBUG)

In [ ]:
day_obs = 20260404
log = logging.getLogger(str(day_obs))

In [ ]:
log.setLevel(logging.DEBUG)

In [ ]:

# Load the nightly table.
# The parquet file was created with: 
# https://github.com/lsst-sitcom/ts_aos_analysis/blob/tickets/DM-54406/notebooks/nightly_report/nightly_report_ts_version.ipynb
parquet_file = f"/home/cslage/MTAOS/times_square_reports/nightly_aos_table_{day_obs}.parquet"
table = pd.read_parquet(parquet_file)
print(f'Loaded {parquet_file}: {len(table)} rows')
print(f'Columns: {sorted(table.columns.tolist())}')

# Load the PID logs
# This was created with PID_Corrections_19Mar26.ipynb
filename = f"/home/cslage/MTAOS/times_square_reports/PID_data_{day_obs}.pkl"
with open(filename, 'rb') as f:
    pidDict = pkl.load(f)


In [ ]:
efd_client = lsst_efd_client.EfdClient("summit_efd")

In [ ]:
start = Time(
    "2026-04-04T23:00:00",
    format="isot",
    scale="utc",
)
end = Time(
    "2026-04-05T10:00:00",
    format="isot",
    scale="utc",
)

In [ ]:
wavefront_errors = await efd_client.select_time_series(
    "lsst.sal.MTAOS.logevent_wavefrontError",
    [
        f"nollZernikeValues{i}" for i in range(27)
    ] + [
        f"nollZernikeIndices{i}" for i in range(27)
    ] + [
        "sensorId",
        "visitId"
    ],
    start=start,
    end=end,
)

In [ ]:
degrees_of_freedom = await efd_client.select_time_series(
    "lsst.sal.MTAOS.logevent_degreeOfFreedom",
    [
        f"aggregatedDoF{i}" for i in range(50)
    ] + [
        f"visitDoF{i}" for i in range(50)
    ] + [
        "visitId",
    ] + [
        f"kpGain{i}" for i in range(50)
    ]+ [
        f"kiGain{i}" for i in range(50)
    ],
    start=start,
    end=end,
)

In [ ]:
def find_visit_index(visit, data):
    visit_ids = np.array(data.visitId.array)
    return np.where(visit_ids == visit)[0][0]

In [ ]:
def find_visit_indices(visit, data):
    visit_ids = np.array(data.visitId.array)
    return np.where(visit_ids == visit)[0]

In [ ]:
def extract_wfe(data):
    wfe = np.zeros(23)
    for i in range(27):
        value = data[f"nollZernikeValues{i}"]
        index = data[f"nollZernikeIndices{i}"]
        if value is not None:
            wfe[index-4] = value
    return wfe

In [ ]:
ofc_data = OFCData(
    name="lsst",
    log=log,
    config_dir="/net/obs-env/auto_base_packages/ts_config_mttcs/MTAOS/ofc"
)

In [ ]:
sensor_ids=np.array([191, 195, 199, 203])
filter_name='i'
rotation_angle=0.0

In [ ]:
ofc = OFC(
    ofc_data=ofc_data,
    log=log,
)

In [ ]:
new_comp_dof_idx = dict(
    m2HexPos=np.ones(5, dtype=bool),
    camHexPos=np.ones(5, dtype=bool),
    M1M3Bend=np.ones(20, dtype=bool),
    M2Bend=np.ones(20, dtype=bool),
)

new_comp_dof_idx["M1M3Bend"][7:] = False
new_comp_dof_idx["M2Bend"][5:] = False

new_comp_dof_idx = dict(
    m2HexPos=np.ones(5, dtype=bool),
    camHexPos=np.ones(5, dtype=bool),
    M1M3Bend=np.ones(20, dtype=bool),
    M2Bend=np.ones(20, dtype=bool),
)

new_comp_dof_idx["M1M3Bend"][7:] = False
new_comp_dof_idx["M2Bend"][5:] = False

ofc.set_truncation_index(12)
ofc_data.zn_selected = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26])
ofc.ofc_data.comp_dof_idx = new_comp_dof_idx
ofc.controller.reset_history()
ofc.state_estimator.refresh_from_ofc_data()

In [ ]:
ofc.controller.kp = 0.3
ofc.controller.ki = 0
ofc.controller.kd = 0
ofc.controller.reset_history()

In [ ]:
valid_visit_ids = np.unique(
    np.array(list(set(degrees_of_freedom.visitId).intersection(set(wavefront_errors.visitId))))
)

In [ ]:
visit_id

In [ ]:
this_table = table[table['seq']==int(visit_id - 1e5*day_obs)]

In [ ]:
rotation_angle = this_table['rotation_angle'].values[0]

In [ ]:
filter_name = list(this_table['band'].values[0])[0]

In [ ]:
derived_lv_dof = []

for visit_id in valid_visit_ids:
    this_table = table[table['seq']==int(visit_id - 1e5*day_obs)]
    rotation_angle = this_table['rotation_angle'].values[0]
    filter_name = list(this_table['band'].values[0])[0]
    index0 = find_visit_index(visit_id, wavefront_errors)
    wavefront_error = np.array(
        [
            extract_wfe(wavefront_errors.iloc[index0]),
            extract_wfe(wavefront_errors.iloc[index0+1]),
            extract_wfe(wavefront_errors.iloc[index0+2]),
            extract_wfe(wavefront_errors.iloc[index0+3]),
        ]
    )
    gain_index = find_visit_index(visit_id, degrees_of_freedom)
    if degrees_of_freedom.iloc[gain_index].kpGain0 > 0.5:
        print(f"Skip {visit_id} with gain {degrees_of_freedom.iloc[gain_index].kpGain0}...")
        continue
    ofc.controller.kp = degrees_of_freedom.iloc[gain_index].kpGain0
    ofc.controller.reset_history()
    print(visit_id)
    m2hex_1, camhex_1, m1m3_, m2_1 = ofc.calculate_corrections(
        wfe=wavefront_error,
        sensor_ids=sensor_ids,
        filter_name=filter_name,
        rotation_angle=rotation_angle,
        subtract_intrinsics=False,
        control_vmodes=False,
    )
    derived_lv_dof.append(ofc.lv_dof)
derived_lv_dof = np.array(derived_lv_dof)
derived_lv_dof_transp = derived_lv_dof.T

In [ ]:
published_dof = []
visit_ids = []
for visit_id in valid_visit_ids:
    index0 = find_visit_index(visit_id, degrees_of_freedom)
    if degrees_of_freedom.iloc[index0].kpGain0 > 0.5:
        print(f"Skip {visit_id} with gain {degrees_of_freedom.iloc[gain_index].kpGain0}...")
        continue
    print(visit_id, degrees_of_freedom.iloc[index0].visitDoF0, degrees_of_freedom.iloc[index0].visitDoF5)
    dofs = []
    for i in range(50):
        dofs.append(degrees_of_freedom.iloc[index0][f"visitDoF{i}"])
    published_dof.append(np.array(dofs))
    visit_ids.append(visit_id)
published_dof = np.array(published_dof)
published_dof = published_dof.T

In [ ]:
print(published_dof.shape, derived_lv_dof_transp.shape)

In [ ]:
dof_labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
     'Cam dz', 'Cam dx', 'Cam dy', 'Cam rx', 'Cam ry',
     'M1M3-1', 'M1M3-2', 'M1M3-3', 'M1M3-4', 'M1M3-5',
     'M1M3-6', 'M1M3-7', 'M1M3-8', 'M1M3-9', 'M1M3-10',
     'M1M3-11', 'M1M3-12', 'M1M3-13', 'M1M3-14', 'M1M3-15',
     'M1M3-16', 'M1M3-17', 'M1M3-18', 'M1M3-19', 'M1M3-20',
     'M2-1', 'M2-2', 'M2-3', 'M2-4', 'M2-5',
     'M2-6', 'M2-7', 'M2-8', 'M2-9', 'M2-10',
     'M2-11', 'M2-12', 'M2-13', 'M2-14', 'M2-15',
     'M2-16', 'M2-17', 'M2-18', 'M2-19', 'M2-20'
     ]


In [ ]:
indices = list(range(0, 17)) + list(range(30,35))
fig, axs = plt.subplots(5,5, figsize=(15, 15))
plt.subplots_adjust(wspace=1.0, hspace=1.0)
plt.suptitle(f"{day_obs}", y=0.95)
axes = []
for i in range(5):
    for j in range(5):
        n = j + 5 * i
        axes.append(axs[i][j])
        if n > 21:
            axs[i][j].set_visible(False)
        
for n, i in enumerate(indices):
    x = published_dof[i,:]
    y = derived_lv_dof_transp[i,:]
    maxlim = max(np.nanpercentile(x, 95), np.nanpercentile(y, 95))
    minlim = min(np.nanpercentile(x, 5), np.nanpercentile(y, 5))
    axes[n].plot([minlim, maxlim],[minlim, maxlim], ls='--', color='red')
    axes[n].set_xlim(minlim, maxlim)
    axes[n].set_ylim(minlim, maxlim)
    axes[n].scatter(x, y, marker='.')
    axes[n].set_title(dof_labels[i])
    axes[n].set_xlabel("Published DoF")
    axes[n].set_ylabel("Calculated DoF")
fig.savefig(f"/home/cslage/MTAOS/times_square_reports/DoF_Check_All_Gains_Rot_Filter_{day_obs}.png")

In [ ]:
for i in range(135):
    x = published_dof[0,i]
    y = derived_lv_dof_transp[0,i]
    print(visit_ids[i], abs(x / y))